In [1]:
!pip install torch torchvision matplotlib opencv-python tqdm scikit-learn

In [2]:
import os
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
class SaliencyDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_list):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = image_list

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.resize(image, (128,128))
        image = image / 255.0

        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (128,128))
        mask = mask / 255.0


        if random.random() > 0.5:
            image = cv2.flip(image, 1)
            mask = cv2.flip(mask, 1)

        image = torch.tensor(image).permute(2,0,1).float()
        mask = torch.tensor(mask).unsqueeze(0).float()

        return image, mask

In [ ]:
image_dir = "/content/drive/MyDrive/saliency/images"
mask_dir = "/content/drive/MyDrive/saliency/masks"

images = os.listdir(image_dir)

train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

train_dataset = SaliencyDataset(image_dir, mask_dir, train_imgs)
val_dataset   = SaliencyDataset(image_dir, mask_dir, val_imgs)
test_dataset  = SaliencyDataset(image_dir, mask_dir, test_imgs)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [6]:
class SimpleSOD(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64,32,2,stride=2),
            nn.ReLU()
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(32,16,2,stride=2),
            nn.ReLU()
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(16,1,2,stride=2),
            nn.Sigmoid()
        )

    def forward(self,x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)

        return x

In [7]:
import torch.nn.functional as F

def iou(pred, target):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def iou_loss(pred, target):
    return 1 - iou(pred, target)

def total_loss(pred, target):
    bce = F.binary_cross_entropy(pred, target)
    return bce + 0.5 * iou_loss(pred, target)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleSOD().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_loss = float('inf')

for epoch in range(15):
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = total_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    #validimi
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = total_loss(preds, masks)
            val_loss += loss.item()

    print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")

In [9]:
def visualize(image, mask, pred):
    image = image.permute(1,2,0).cpu().numpy()
    mask = mask.squeeze().cpu().numpy()
    pred = pred.squeeze().cpu().numpy()

    plt.figure(figsize=(12,4))

    plt.subplot(1,4,1)
    plt.title("Input")
    plt.imshow(image)

    plt.subplot(1,4,2)
    plt.title("Ground Truth")
    plt.imshow(mask, cmap='gray')

    plt.subplot(1,4,3)
    plt.title("Prediction")
    plt.imshow(pred, cmap='gray')

    plt.subplot(1,4,4)
    plt.title("Overlay")
    plt.imshow(image)
    plt.imshow(pred, alpha=0.5, cmap='jet')

    plt.show()

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

imgs, masks = next(iter(test_loader))

imgs = imgs.to(device)
preds = model(imgs)

for i in range(3):
    visualize(imgs[i].cpu(), masks[i], preds[i].detach().cpu())

In [11]:
class ImprovedSOD(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(16,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.dec1 = nn.ConvTranspose2d(64,32,2,stride=2)
        self.dec2 = nn.ConvTranspose2d(32,16,2,stride=2)
        self.dec3 = nn.ConvTranspose2d(16,1,2,stride=2)

    def forward(self,x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = torch.relu(self.dec1(x))
        x = torch.relu(self.dec2(x))
        x = torch.sigmoid(self.dec3(x))

        return x

In [12]:
def predict_image(path):
    img = cv2.imread(path)
    img_resized = cv2.resize(img,(128,128))/255.0

    tensor = torch.tensor(img_resized).permute(2,0,1).unsqueeze(0).float().to(device)

    pred = model(tensor)[0]

    visualize(tensor[0].cpu(), torch.zeros_like(pred), pred)